# 🎬 MiniMax H3 — Reference-to-Video + Audio · Gradio Live
**Cell 1**: Cài đặt ComfyUI + tải models → **Cell 2**: Gradio UI (gradio.live)

Models từ `Comfy-Org/MiniMax-H3` (HuggingFace, public, KHÔNG cần token)

In [ ]:
#@title 📦 Cell 1: Install ComfyUI + Download MiniMax H3 Models
import os

COMFYUI_ROOT = "/content/ComfyUI"

# ==========================================================================
# Models
# ==========================================================================
MINIMAX_MODELS = [
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors",
     f"{COMFYUI_ROOT}/models/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
     f"{COMFYUI_ROOT}/models/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors",
     f"{COMFYUI_ROOT}/models/vae/minimax_h3_video_vae_fp16.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors",
     f"{COMFYUI_ROOT}/models/vae/minimax_h3_audio_vae_fp32.safetensors"),
    ("https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
     f"{COMFYUI_ROOT}/models/loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"),
]

def download_models(models):
    has_aria2 = os.system("which aria2c > /dev/null 2>&1") == 0
    for url, dest in models:
        if os.path.exists(dest) and os.path.getsize(dest) > 1024 * 1024 * 50:
            size_mb = os.path.getsize(dest) / 1024 / 1024
            print(f"✅ Đã có ({size_mb:.0f} MB): {os.path.basename(dest)}")
            continue
        dest_dir = os.path.dirname(dest)
        dest_name = os.path.basename(dest)
        os.makedirs(dest_dir, exist_ok=True)
        print(f"⬇️  Đang tải: {dest_name} ...")
        if has_aria2:
            cmd = f'aria2c -c -x 16 -s 16 -k 1M -d "{dest_dir}" -o "{dest_name}" "{url}"'
        else:
            cmd = f'wget -q --show-progress -c "{url}" -O "{dest}"'
        exit_code = os.system(cmd)
        if exit_code == 0 and os.path.exists(dest):
            size_mb = os.path.getsize(dest) / 1024 / 1024
            print(f"✅ Xong ({size_mb:.0f} MB): {dest_name}")
        else:
            print(f"❌ Lỗi tải: {dest_name}")

def setup_comfyui(target_dir="/content/ComfyUI"):
    main_py = os.path.join(target_dir, "main.py")
    if os.path.exists(main_py):
        print("✅ ComfyUI đã sẵn sàng")
        return True
    print("🔄 Đang thiết lập ComfyUI...")
    if not os.path.exists(target_dir):
        os.system(f"git clone https://github.com/comfyanonymous/ComfyUI {target_dir}")
    else:
        temp_dir = "/content/temp_comfyui"
        os.system(f"rm -rf {temp_dir}")
        os.system(f"git clone https://github.com/comfyanonymous/ComfyUI {temp_dir}")
        if os.path.exists(temp_dir):
            os.system(f"cp -rn {temp_dir}/* {target_dir}/ 2>/dev/null || true")
            os.system(f"cp -rn {temp_dir}/.* {target_dir}/ 2>/dev/null || true")
            os.system(f"rm -rf {temp_dir}")
    req_path = os.path.join(target_dir, "requirements.txt")
    if os.path.exists(req_path):
        os.system(f"pip install -q -r {req_path}")
    return os.path.exists(main_py)

print("=" * 60)
print("  MiniMax H3 — Cài Đặt & Tải Model (Cell 1/2)")
print("=" * 60)

os.system("apt-get -y install -qq aria2 > /dev/null 2>&1 || true")
setup_comfyui(COMFYUI_ROOT)

print("📦 Cài thư viện...")
os.system("pip install -q uv")
os.system("uv pip install -q --system gradio opencv-python accelerate diffusers einops sentencepiece av spandrel aiohttp || pip install -q gradio opencv-python accelerate diffusers einops sentencepiece av spandrel aiohttp")

download_models(MINIMAX_MODELS)
print("\n🎉 HOÀN TẤT CELL 1! Chạy Cell 2 để mở giao diện.")

In [ ]:
#@title 🎬 Cell 2: MiniMax H3 — Gradio Live UI
get_ipython().system("pip install -q gradio opencv-python")

import glob
import json
import os
import random
import re
import shutil
import socket
import subprocess
import time
import urllib.request

import gradio as gr

# ==========================================================================
# CONFIG
# ==========================================================================
INPUT_DIR  = "/content/ComfyUI/input/"
OUTPUT_DIR = "/content/ComfyUI/output/"
COMFYUI_DIR = "/content/ComfyUI"
COMFYUI_LOG_PATH = "/content/comfyui.log"

UNET_FILENAME       = "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
CLIP_FILENAME       = "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"
VIDEO_VAE_FILENAME  = "minimax_h3_video_vae_fp16.safetensors"
AUDIO_VAE_FILENAME  = "minimax_h3_audio_vae_fp32.safetensors"
TURBO_LORA_FILENAME = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"

# ==========================================================================
# SERVER MANAGEMENT
# ==========================================================================
_SERVER_STATE = {"running_low_vram": None, "custom_nodes_mtime": None}

def is_server_running(port=8188):
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{port}/system_stats")
        with urllib.request.urlopen(req, timeout=1.5) as resp:
            return resp.status == 200
    except Exception:
        pass
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1.0)
            return s.connect_ex(("127.0.0.1", port)) == 0
    except Exception:
        return False

def _get_custom_nodes_mtime():
    cn_dir = os.path.join(COMFYUI_DIR, "custom_nodes")
    try:
        mtimes = []
        for entry in os.scandir(cn_dir):
            mtimes.append(entry.stat().st_mtime)
            if entry.is_dir():
                try:
                    for sub in os.scandir(entry.path):
                        if sub.name.endswith(".py"):
                            mtimes.append(sub.stat().st_mtime)
                except OSError:
                    pass
        return max(mtimes) if mtimes else 0.0
    except OSError:
        return 0.0

def ensure_server(low_vram=True, boot_timeout=120):
    main_py = os.path.join(COMFYUI_DIR, "main.py")
    if not os.path.exists(main_py):
        raise RuntimeError("❌ /content/ComfyUI/main.py không tồn tại! Chạy lại Cell 1.")

    current_mtime = _get_custom_nodes_mtime()
    if is_server_running():
        if _SERVER_STATE["running_low_vram"] is None or (
            _SERVER_STATE["running_low_vram"] == low_vram
            and _SERVER_STATE["custom_nodes_mtime"] == current_mtime
        ):
            _SERVER_STATE["running_low_vram"] = low_vram
            _SERVER_STATE["custom_nodes_mtime"] = current_mtime
            return

    os.system("fuser -k 8188/tcp 2>/dev/null || true")
    os.system("pkill -9 -f 'python.*main.py' 2>/dev/null || true")
    time.sleep(2)

    os.chdir(COMFYUI_DIR)
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    log_out = open(COMFYUI_LOG_PATH, "w", encoding="utf-8", errors="ignore")

    cmd = ["python", "-u", "main.py", "--listen", "127.0.0.1", "--port", "8188"]
    if low_vram:
        cmd.append("--lowvram")
        cmd.append("--cache-none")

    proc = subprocess.Popen(cmd, cwd=COMFYUI_DIR, stdout=log_out, stderr=subprocess.STDOUT)

    waited = 0
    while not is_server_running():
        time.sleep(2)
        waited += 2
        ret = proc.poll()
        if ret is not None:
            log_out.flush(); log_out.close()
            try:
                with open(COMFYUI_LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
                    tail = "".join(f.readlines()[-35:])
            except Exception:
                tail = "Không đọc được log."
            raise RuntimeError(f"❌ ComfyUI crash (exit {ret})!\n{tail}")
        if waited > boot_timeout:
            log_out.flush(); log_out.close()
            try:
                with open(COMFYUI_LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
                    tail = "".join(f.readlines()[-35:])
            except Exception:
                tail = "Không đọc được log."
            raise RuntimeError(f"❌ Server timeout {boot_timeout}s!\n{tail}")

    _SERVER_STATE["running_low_vram"] = low_vram
    _SERVER_STATE["custom_nodes_mtime"] = current_mtime

def force_restart_server():
    os.system("fuser -k 8188/tcp 2>/dev/null || true")
    os.system("pkill -9 -f 'python.*main.py' 2>/dev/null || true")
    time.sleep(2)
    _SERVER_STATE["running_low_vram"] = None
    _SERVER_STATE["custom_nodes_mtime"] = None
    return "🟢 Server đã tắt. Lần tạo video tiếp sẽ tự khởi động lại."

def free_comfyui_memory():
    for endpoint, payload in [
        ("interrupt", b"{}"),
        ("queue", json.dumps({"clear": True}).encode()),
        ("free", json.dumps({"unload_models": True, "free_memory": True}).encode()),
    ]:
        try:
            req = urllib.request.Request(
                f"http://127.0.0.1:8188/{endpoint}", data=payload,
                headers={"Content-Type": "application/json"})
            urllib.request.urlopen(req, timeout=5)
        except Exception:
            pass
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return "🟢 Đã giải phóng GPU VRAM!"

def read_server_log():
    if not os.path.exists(COMFYUI_LOG_PATH):
        return "ℹ️ Chưa có log."
    try:
        with open(COMFYUI_LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
            return "".join(f.readlines()[-40:]) or "ℹ️ Log rỗng."
    except Exception as e:
        return f"⚠️ Lỗi đọc log: {e}"

# ==========================================================================
# HELPERS
# ==========================================================================
def get_seed(v):
    try:
        v = int(v)
    except Exception:
        v = -1
    return random.randint(1, 999_999_999) if v == -1 else v

def get_comfyui_progress_line():
    if not os.path.exists(COMFYUI_LOG_PATH):
        return ""
    try:
        with open(COMFYUI_LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
        for l in reversed(lines[-20:]):
            s = l.strip()
            if "%" in s or "it/s" in s or "s/it" in s:
                return re.sub(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])', '', s)[:100]
            if "Executing node" in s:
                return re.sub(r'\x1B(?:[@-Z\\-_]|\[[0-?]*[ -/]*[@-~])', '', s)[:100]
    except Exception:
        pass
    return ""

def parse_resolution(ratio_str):
    if "864x480"   in ratio_str: return 864,  480
    if "480x864"   in ratio_str: return 480,  864
    if "1344x768"  in ratio_str: return 1344, 768
    if "768x1344"  in ratio_str: return 768,  1344
    if "1056x1056" in ratio_str: return 1056, 1056
    return 864, 480

def duration_to_length(duration_s, fps):
    raw = int(round(float(duration_s) * float(fps)))
    n = max(1, round((raw - 1) / 4))
    return 4 * n + 1

def split_prompts(text):
    if not text:
        return []
    parts = re.split(r'\n\s*\n', text.strip())
    return [p.strip() for p in parts if p.strip()]

def count_scenes(text):
    return f"🔹 **Số phân cảnh:** {len(split_prompts(text))}"

def find_latest_video(output_dir=OUTPUT_DIR):
    candidates = []
    for pat in ("**/*.mp4", "**/*.webm", "**/*.mkv"):
        candidates.extend(glob.glob(os.path.join(output_dir, pat), recursive=True))
    return max(candidates, key=os.path.getmtime) if candidates else None

def concat_videos(video_paths, output_name):
    list_path = os.path.join(OUTPUT_DIR, f"{output_name}_list.txt")
    with open(list_path, "w", encoding="utf-8") as f:
        for v in video_paths:
            f.write(f"file '{os.path.abspath(v)}'\n")
    output_path = os.path.join(OUTPUT_DIR, f"{output_name}.mp4")
    cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_path, "-c", "copy", output_path]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(output_path):
        cmd2 = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_path,
                "-c:v", "libx264", "-preset", "slow", "-crf", "18",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", output_path]
        result2 = subprocess.run(cmd2, capture_output=True, text=True)
        if result2.returncode != 0:
            raise RuntimeError(f"ffmpeg concat lỗi: {result2.stderr[-500:]}")
    try:
        os.remove(list_path)
    except Exception:
        pass
    return output_path

# ==========================================================================
# SUBMIT & POLL (generator with progress)
# ==========================================================================
def submit_and_wait_gen(workflow, scene_label="", max_wait_seconds=1800, poll_interval=3):
    data = json.dumps({"prompt": workflow}).encode("utf-8")
    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    try:
        response = urllib.request.urlopen(req, timeout=30)
        prompt_id = json.loads(response.read())["prompt_id"]
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="ignore")
        free_comfyui_memory()
        raise RuntimeError(f"ComfyUI từ chối workflow: {body[:800]}")
    except Exception as e:
        free_comfyui_memory()
        raise RuntimeError(f"Lỗi gửi job API: {e}")

    waited = 0
    consecutive_errors = 0
    GRACE_POLLS = 3
    poll_count = 0

    while waited < max_wait_seconds:
        try:
            history = json.loads(urllib.request.urlopen(
                urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}"),
                timeout=30).read())
            consecutive_errors = 0
            if str(prompt_id) in history:
                yield True, prompt_id, "Hoàn tất"
                return

            queue = json.loads(urllib.request.urlopen(
                urllib.request.Request("http://127.0.0.1:8188/queue"), timeout=30).read())
            is_running = any(
                str(job[1]) == str(prompt_id)
                for job in queue.get("queue_running", []) + queue.get("queue_pending", [])
            )
            poll_count += 1
            if not is_running and poll_count > GRACE_POLLS:
                free_comfyui_memory()
                raise RuntimeError(f"Render thất bại ở {scene_label}")

            p_line = get_comfyui_progress_line()
            m, s = divmod(waited, 60)
            prog = f"[{m:02d}m{s:02d}s] {p_line}" if p_line else f"[{m:02d}m{s:02d}s] Đang tính toán..."
            yield False, prompt_id, prog

        except RuntimeError:
            free_comfyui_memory()
            raise
        except Exception:
            consecutive_errors += 1
            if consecutive_errors >= 15:
                free_comfyui_memory()
                raise RuntimeError(f"Server không phản hồi ở {scene_label}!")

        time.sleep(poll_interval)
        waited += poll_interval

    free_comfyui_memory()
    raise RuntimeError(f"Timeout: {scene_label} quá {max_wait_seconds // 60} phút!")

# ==========================================================================
# BUILD WORKFLOW (MiniMax H3 ref2va — 4 ảnh tham khảo)
# ==========================================================================
def build_minimax_workflow(
    *, prompt, width=864, height=480, duration_s=5.0, fps=24,
    seed=None, scheduler="beta", steps_full=20, steps_turbo=4,
    use_turbo_lora=False, ref_image_size="match",
    pic1_name=None, pic2_name=None, pic3_name=None, pic4_name=None,
):
    if seed is None:
        seed = random.randint(1, 999_999_999)
    length = duration_to_length(duration_s, fps)

    wf = {
        "unet": {"class_type": "UNETLoader", "inputs": {
            "unet_name": UNET_FILENAME, "weight_dtype": "default"}},
        "clip": {"class_type": "CLIPLoader", "inputs": {
            "clip_name": CLIP_FILENAME, "type": "minimax", "device": "default"}},
        "video_vae": {"class_type": "VAELoader", "inputs": {
            "vae_name": VIDEO_VAE_FILENAME}},
        "audio_vae": {"class_type": "VAELoader", "inputs": {
            "vae_name": AUDIO_VAE_FILENAME}},
        "turbo_lora": {"class_type": "LoraLoaderModelOnly", "inputs": {
            "model": ["unet", 0], "lora_name": TURBO_LORA_FILENAME, "strength_model": 1.0}},
        "model_switch": {"class_type": "ComfySwitchNode", "inputs": {
            "on_false": ["unet", 0], "on_true": ["turbo_lora", 0],
            "switch": bool(use_turbo_lora)}},
        "steps_full_node":  {"class_type": "PrimitiveInt", "inputs": {"value": steps_full}},
        "steps_turbo_node": {"class_type": "PrimitiveInt", "inputs": {"value": steps_turbo}},
        "steps_switch": {"class_type": "ComfySwitchNode", "inputs": {
            "on_false": ["steps_full_node", 0], "on_true": ["steps_turbo_node", 0],
            "switch": bool(use_turbo_lora)}},
        "noise": {"class_type": "RandomNoise", "inputs": {"noise_seed": int(seed)}},
        "ref2va": {
            "class_type": "MiniMaxH3ReferenceToVideo",
            "inputs": {
                "clip": ["clip", 0], "vae": ["video_vae", 0],
                "audio_vae": ["audio_vae", 0], "prompt": prompt,
                "width": int(width), "height": int(height),
                "length": int(length), "ref_image_size": ref_image_size,
            },
        },
        "sampler_sel": {"class_type": "KSamplerSelect", "inputs": {
            "sampler_name": "res_multistep"}},
        "scheduler_node": {"class_type": "BasicScheduler", "inputs": {
            "model": ["model_switch", 0], "scheduler": scheduler,
            "steps": ["steps_switch", 0], "denoise": 1.0}},
        "guider": {"class_type": "BasicGuider", "inputs": {
            "model": ["model_switch", 0], "conditioning": ["ref2va", 0]}},
        "sampler": {"class_type": "SamplerCustomAdvanced", "inputs": {
            "noise": ["noise", 0], "guider": ["guider", 0],
            "sampler": ["sampler_sel", 0], "sigmas": ["scheduler_node", 0],
            "latent_image": ["ref2va", 1]}},
        "vae_decode": {"class_type": "VAEDecode", "inputs": {
            "samples": ["sampler", 0], "vae": ["video_vae", 0]}},
        "audio_decode": {"class_type": "VAEDecodeAudio", "inputs": {
            "samples": ["sampler", 0], "vae": ["audio_vae", 0]}},
        "create_video": {"class_type": "CreateVideo", "inputs": {
            "images": ["vae_decode", 0], "audio": ["audio_decode", 0], "fps": fps}},
        "save_video": {"class_type": "SaveVideo", "inputs": {
            "video": ["create_video", 0], "filename_prefix": "video/MiniMax_H3",
            "format": "auto", "codec": "auto"}},
    }

    ref_slots = [
        ("ref_images.ref_image_0", pic1_name),
        ("ref_images.ref_image_1", pic2_name),
        ("ref_images.ref_image_2", pic3_name),
        ("ref_images.ref_image_3", pic4_name),
    ]
    for slot_key, img_name in ref_slots:
        if img_name:
            node_id = f"load_{slot_key.split('.')[-1]}"
            wf[node_id] = {"class_type": "LoadImage", "inputs": {"image": img_name}}
            wf["ref2va"]["inputs"][slot_key] = [node_id, 0]

    return wf

# ==========================================================================
# GENERATE — GRADIO GENERATOR
# ==========================================================================
def generate_minimax_gradio(
    pic1_path, pic2_path, pic3_path, pic4_path,
    prompt_main,
    aspect_ratio, duration_s, fps, seed_val, num_segments, fixed_seed,
    scheduler, use_turbo_lora, ref_image_size,
    low_vram,
):
    if not pic1_path:
        yield None, None, "⚠️ Tải ít nhất Pic 1!"; return

    prompts = split_prompts(prompt_main)
    if not prompts:
        yield None, None, "⚠️ Nhập ít nhất 1 prompt!"; return

    width, height = parse_resolution(aspect_ratio)

    yield None, None, "🔄 Khởi động ComfyUI server..."
    try:
        ensure_server(low_vram)
    except Exception as e:
        yield None, None, f"❌ {e}"; return

    base_seed = get_seed(seed_val)
    os.makedirs(INPUT_DIR, exist_ok=True)

    def _copy_img(path, slot):
        if not path:
            return None
        ext = os.path.splitext(path)[1].lower() or ".png"
        name = f"minimax_{slot}_{int(time.time())}{ext}"
        shutil.copy(path, os.path.join(INPUT_DIR, name))
        return name

    pic1 = _copy_img(pic1_path, "pic1")
    pic2 = _copy_img(pic2_path, "pic2")
    pic3 = _copy_img(pic3_path, "pic3")
    pic4 = _copy_img(pic4_path, "pic4")
    loaded = [s for s in [pic1, pic2, pic3, pic4] if s]

    if len(prompts) > 1:
        scene_prompts = prompts
    else:
        num_segments = max(1, int(num_segments))
        scene_prompts = [prompts[0]] * num_segments

    total = len(scene_prompts)
    total_s = total * float(duration_s)
    mode = "Turbo 4-step" if use_turbo_lora else "Full 20-step"

    yield None, None, (
        f"✅ Server sẵn sàng. Render {total} cảnh (~{total_s:.0f}s)...\n"
        f"📸 {len(loaded)} ảnh · {mode} · Seed: {base_seed} · {scheduler}"
    )

    generated = []
    for i, p in enumerate(scene_prompts):
        label = f"cảnh {i+1}/{total}"
        seed_i = base_seed if fixed_seed else (base_seed + i)

        yield generated, None, (
            f"🔄 Render {label} [{mode}] (Seed: {seed_i})\n"
            f"📝 {p[:120]}..."
        )

        wf = build_minimax_workflow(
            prompt=p, width=width, height=height,
            duration_s=float(duration_s), fps=int(fps), seed=seed_i,
            scheduler=scheduler, use_turbo_lora=bool(use_turbo_lora),
            ref_image_size=ref_image_size,
            pic1_name=pic1, pic2_name=pic2, pic3_name=pic3, pic4_name=pic4,
        )

        timeout = max(900, int(duration_s) * 150) if use_turbo_lora else max(2700, int(duration_s) * 400)

        try:
            for is_done, p_id, prog_msg in submit_and_wait_gen(wf, scene_label=label, max_wait_seconds=timeout):
                if not is_done:
                    yield generated, None, (
                        f"🔄 Render {label} [{mode}] (Seed: {seed_i})\n"
                        f"⏳ {prog_msg}\n"
                        f"📝 {p[:120]}..."
                    )
                else:
                    break
        except RuntimeError as e:
            free_comfyui_memory()
            yield generated, None, f"❌ {e}"; return

        latest = find_latest_video()
        if not latest:
            yield generated, None, f"⚠️ Không tìm thấy video ở {label}!"; return

        generated.append(latest)
        yield generated, None, f"🔔 [DING] ✅ Xong {label}!"

    if len(generated) > 1:
        yield generated, None, "🔄 Ghép nối các cảnh..."
        try:
            final = concat_videos(generated, f"final_minimax_{int(time.time())}")
        except Exception as e:
            yield generated, generated[-1], f"⚠️ {e}"; return
        yield generated, final, (
            f"🔔 [DING] 🎉 Hoàn tất! ({total_s:.0f}s, {total} cảnh) Seed: {base_seed}"
        )
    elif len(generated) == 1:
        yield generated, generated[0], (
            f"🔔 [DING] 🎉 Render xong! ({duration_s}s) Seed: {base_seed}"
        )

# ==========================================================================
# GRADIO UI
# ==========================================================================
ratio_choices = [
    "16:9 (864x480) ⭐ Khuyên dùng (nhanh, nhẹ VRAM)",
    "9:16 (480x864) ⭐ TikTok/Reels",
    "16:9 (1344x768) HD — Chậm, cần >25 phút/cảnh",
    "9:16 (768x1344) HD dọc — Chậm",
    "1:1  (1056x1056) Vuông",
]

custom_css = """
.gradio-container { max-width: 1560px; margin: 0 auto; }
#mm-header { background: linear-gradient(135deg, #0f4c81 0%, #1a73e8 100%);
             border-radius:16px; padding:20px 26px; margin-bottom:14px; }
#mm-header h1, #mm-header p, #mm-header h3, #mm-header div { color:#fff !important; margin:0 !important; }
.info-box { background:rgba(26,115,232,.08); border-left:3px solid #1a73e8;
            padding:10px 14px; border-radius:8px; font-size:.87rem; margin-bottom:8px; }
.scene-counter { display:inline-block; background:rgba(26,115,232,.12); padding:6px 14px;
                 border-radius:999px; font-weight:600 !important; font-size:.85rem !important; }
.scene-counter p { margin:0 !important; color:#1a73e8 !important; }
.status-box textarea { font-family:monospace !important; font-size:.82rem !important; }
"""

notification_js = """
function(){
    let last="";
    function ding(){
        try{
            let c=new(window.AudioContext||window.webkitAudioContext)();
            let o=c.createOscillator(),g=c.createGain();
            o.type='sine'; o.frequency.setValueAtTime(880,c.currentTime);
            o.frequency.exponentialRampToValueAtTime(1760,c.currentTime+.15);
            g.gain.setValueAtTime(.3,c.currentTime);
            g.gain.exponentialRampToValueAtTime(.01,c.currentTime+.4);
            o.connect(g);g.connect(c.destination);o.start();o.stop(c.currentTime+.4);
        }catch(e){}
    }
    new MutationObserver(()=>{
        document.querySelectorAll('.status-box textarea').forEach(tb=>{
            let t=tb.value||"";
            if(t.includes('[DING]')&&t!==last){last=t;ding();}
        });
    }).observe(document.body,{childList:true,subtree:true,characterData:true});
}
"""

with gr.Blocks(
    title="MiniMax H3 Studio",
    fill_width=True,
) as demo:

    with gr.Column(elem_id="mm-header"):
        with gr.Row():
            with gr.Column(scale=4):
                gr.Markdown(
                    "### 🎬 MiniMax H3 — Reference-to-Video + Audio\n"
                    "Tạo video có audio native từ ảnh tham khảo · Tối đa 4 ảnh · 2K 24fps ~15s/clip\n\n"
                    "🎭 Dùng `<Picture 1>` ... `<Picture 4>` trong prompt · 🔊 Audio sinh native"
                )
            with gr.Column(scale=1, min_width=160):
                restart_btn = gr.Button("🔄 Restart Server", size="sm")
                free_btn = gr.Button("🧹 Giải Phóng VRAM", size="sm")
                log_btn = gr.Button("📋 Xem Log", size="sm")
                restart_out = gr.Markdown("🟢 Sẵn sàng")
        restart_btn.click(fn=force_restart_server, outputs=[restart_out])
        free_btn.click(fn=free_comfyui_memory, outputs=[restart_out])

    with gr.Row():
        with gr.Column(scale=5):
            with gr.Group():
                gr.Markdown("### 📸 Ảnh tham khảo")
                gr.Markdown(
                    "<div class='info-box'>"
                    "Upload ảnh rồi tham chiếu bằng <code>&lt;Picture 1&gt;</code> ... <code>&lt;Picture 4&gt;</code> trong prompt. "
                    "Chỉ <b>Pic 1</b> bắt buộc. MiniMax H3 hỗ trợ tối đa 9 ảnh."
                    "</div>"
                )
                with gr.Row():
                    mm_pic1 = gr.Image(label="🎭 Pic 1 (bắt buộc)", type="filepath")
                    mm_pic2 = gr.Image(label="🎭 Pic 2", type="filepath")
                with gr.Row():
                    mm_pic3 = gr.Image(label="🎭 Pic 3", type="filepath")
                    mm_pic4 = gr.Image(label="🎭 Pic 4", type="filepath")

            with gr.Group():
                gr.Markdown("### 📝 Prompt")
                gr.Markdown(
                    "<div class='info-box'>"
                    "Dùng <code>&lt;Picture 1&gt;</code> ... <code>&lt;Picture 4&gt;</code> để tham chiếu ảnh. "
                    "Mỗi cảnh cách nhau <b>1 dòng trống</b>. "
                    "MiniMax H3 sinh <b>audio native</b> — mô tả âm thanh trực tiếp trong prompt!"
                    "</div>"
                )
                scene_display = gr.Markdown("🔹 **Số phân cảnh:** 0", elem_classes="scene-counter")
                mm_prompt = gr.Textbox(
                    label="Prompt (mỗi cảnh cách 1 dòng trống)", lines=8,
                    placeholder=(
                        "<Picture 1> stands on a rooftop at night, cape billowing in wind. "
                        "He says 'Get ready!' with dramatic echo. City neon below.\n\n"
                        "Wide shot — <Picture 1> and <Picture 2> leap off the edge, "
                        "massive explosion behind them, cinematic BOOM sound."
                    ))

            with gr.Accordion("⚙️ Cài đặt nâng cao", open=False):
                gr.Markdown("**📐 Độ phân giải & Thời lượng**")
                ratio_mm = gr.Radio(label="Tỉ lệ", choices=ratio_choices, value=ratio_choices[0])
                with gr.Row():
                    dur_mm = gr.Slider(label="⏱️ Giây/cảnh", minimum=2, maximum=15, step=1, value=5)
                    fps_mm = gr.Number(label="🎞️ FPS", value=24, precision=0, interactive=False)
                with gr.Row():
                    seed_mm = gr.Number(label="🎲 Seed (-1=random)", value=-1, precision=0)
                    seg_mm = gr.Slider(label="🔢 Segments (1 prompt)", minimum=1, maximum=10, step=1, value=1)
                fixed_mm = gr.Checkbox(label="🔗 Cùng Seed mọi cảnh", value=False)

                gr.Markdown("**🧬 Sampler & LoRA**")
                with gr.Row():
                    sched_mm = gr.Radio(label="Scheduler", choices=["beta", "normal", "simple"], value="beta")
                    turbo_mm = gr.Checkbox(
                        label="⚡ Lightning LoRA (4 steps — nhanh x5)", value=True,
                        info="BẬT: ~2-4 phút/cảnh. TẮT: ~25-35 phút, chất lượng cao hơn.")
                ref_mm = gr.Radio(
                    label="ref_image_size", choices=["match", "max"], value="match",
                    info="match = nhanh · max = giữ chi tiết ảnh tốt hơn")

                gr.Markdown("**⚙️ Server**")
                vram_mm = gr.Checkbox(label="🧊 Low VRAM (--cache-none)", value=True)

        with gr.Column(scale=5):
            with gr.Group():
                gallery_mm = gr.Gallery(label="🎥 Các cảnh lẻ", columns=2, height="auto")
                video_mm = gr.Video(label="🎬 Video hoàn chỉnh")
                with gr.Row():
                    btn_mm = gr.Button("🎬 Bắt Đầu Tạo Video", variant="primary", scale=3)
                    clr_mm = gr.Button("🗑️ Clear", scale=1)
                status_mm = gr.Textbox(
                    label="ℹ️ Status", interactive=False, lines=5, elem_classes="status-box")

            gr.Markdown(
                "<div class='info-box'>"
                "<b>💡 Mẹo:</b><br>"
                "• Dùng <code>&lt;Picture N&gt;</code> trong prompt để tham chiếu ảnh.<br>"
                "• Mô tả <b>âm thanh / giọng nói</b> trực tiếp — model sinh audio native.<br>"
                "• <b>Turbo 4-step</b> để preview nhanh, tắt để render full 20-step.<br>"
                "• Mỗi cảnh cách 1 dòng trống → auto render & ghép nối!"
                "</div>"
            )

    mm_prompt.change(fn=count_scenes, inputs=[mm_prompt], outputs=[scene_display])
    btn_mm.click(
        fn=generate_minimax_gradio,
        inputs=[
            mm_pic1, mm_pic2, mm_pic3, mm_pic4,
            mm_prompt,
            ratio_mm, dur_mm, fps_mm, seed_mm, seg_mm, fixed_mm,
            sched_mm, turbo_mm, ref_mm,
            vram_mm,
        ],
        outputs=[gallery_mm, video_mm, status_mm],
    )
    def on_clear():
        free_comfyui_memory()
        return None, None, "🟢 Đã dọn dẹp!", "🔹 **Số phân cảnh:** 0"
    clr_mm.click(fn=on_clear, outputs=[gallery_mm, video_mm, status_mm, scene_display])
    log_btn.click(fn=read_server_log, outputs=[status_mm])

print("🔄 Khởi động ComfyUI server...")
try:
    ensure_server(low_vram=True)
    print("🟢 ComfyUI server sẵn sàng!")
except Exception as e:
    print(f"⚠️ {e}")

demo.queue()
demo.launch(
    share=True, inline=False, debug=True,
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="indigo", neutral_hue="slate"),
    css=custom_css, js=notification_js,
)